# 🕵️ Case file 01 — The Gold bag leak

> **Mission briefing.** It is 09:14 at Skyline Travel and the support queue is on fire. Customers are being quoted baggage rules that are not theirs. You are the Observability Detective on duty. The answers sound confident, so the only way to find out what went wrong is to open the activity logs.

Before answering, the assistant searches a policy database and passes what it finds along with the question. In technical terms this is retrieval-augmented generation (RAG). Every run leaves an **activity log** (Langfuse calls it a *trace*). It is made of **steps** (*spans*): the search step and the answer step (*generation*).

## Case objective
Work out which policies the assistant was allowed to see, and how that changes the answer, its cost and its speed.

## What to expect
Three levels. Each run prints the answer and a link to its activity log. The evidence you need is in the activity log, not in the notebook.

## Bad practice
Trusting a confident answer without checking which policies it was based on.

## Good practice
Check the search step: what was searched, with which limits, and what came back.

## Why the good practice works
The assistant can only be as right as the policies it was given. The search step shows exactly what it was given.

## Deliverable
A claim code for each level you solve, and for your demo, one before/after pair of activity-log links.

## 🎮 How the game works

- Put your team name in the next cell and run it. Use **the same name in every notebook**.
- Each level asks you to find something in the activity logs. Type your answer in `check_flag(...)` and run it. Wrong guesses cost nothing.
- Correct answers give a **claim code**. Paste it into the game master's form.
- Stuck? `hint("1-1")` shows a hint. Each hint costs 25% of that level's points (two hints at most).
- Points: Level 1 = 100, Level 2 = 200, Level 3 = 300.
- Rumour has it a detective left a note somewhere in this case's activity logs. If you find it, submit it as level `1-B`.

In [ ]:
from experiment_utils import check_flag, close_case, hint, run_rag, set_team, show_rag_results

TEAM = "your-team-name"  # ← your team name, the same in every notebook
set_team(TEAM)

## Level 1 — The wrong fare (100 points)

A **Basic**-fare customer asks how many checked bags they get. The assistant answers twice: once searching every policy, once searching only the customer's fare.

**Find:** the id of the policy the search returned when it searched every policy.

In [ ]:
customer = "How many checked bags do I get on my booking?"
runs = [run_rag(customer, None, "no-filter"), run_rag(customer, "basic", "filter")]
show_rag_results(runs)

In [ ]:
check_flag("1-1", "type your answer here")
# hint("1-1")

## Level 2 — Two questions, one answer (200 points)

A **Silver**-fare customer asks two things at once. The search is limited to their fare, so the policies are the right fare. Is the answer complete?

**Find:** the id of the policy the customer needed but the first search never handed to the assistant. Change `top_k` (how many policies the search returns) and rerun the second cell to compare.

In [ ]:
question = "Can I bring my cat in the cabin, and what does a second checked bag cost?"
level2 = [run_rag(question, "silver", "top-1", level=2, query=question)]
show_rag_results(level2)

In [ ]:
top_k = 1  # ← try a bigger number, then compare the two activity logs
level2.append(run_rag(question, "silver", f"top-{top_k}", level=2, query=question, top_k=top_k))
show_rag_results(level2[-1:])

In [ ]:
check_flag("1-2", "type your answer here")
# hint("1-2")

## Level 3 — Playing it safe gets expensive (300 points)

A colleague proposes a shortcut: *"Skip the search and send every policy to the assistant, to be safe."* The answer still looks fine. Compare the two runs: the words the assistant had to read (input tokens), the time it took, and the cost shown in Langfuse.

**Find:** how many policies the shortcut sent to the assistant.

In [ ]:
level3 = [run_rag(customer, "basic", "filter", level=3),
          run_rag(customer, None, "send-all", level=3, send_all=True)]
show_rag_results(level3)

In [ ]:
check_flag("1-3", "type your answer here")
# hint("1-3")

In [ ]:
# Found the detective's note?
check_flag("1-B", "type your answer here")

## 🏁 Wrap-up

Run the last cell to close the case and send the activity logs to Langfuse. Keep one before/after pair of links for your demo.

In [ ]:
close_case("activity-1", runs + level2 + level3)